In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# 데이터보다 feature가 훨씬 많은 고차원 회귀 문제를 만든다.
num_train = 20
num_val = 100

num_inputs = 200 # input 데이터 차원
batch_size = 5

# weights: [200, 1]
true_weights = torch.full((num_inputs, 1), 0.01)
true_bias = 0.05

# features: [120, 200]
features = torch.randn(num_train + num_val, num_inputs)

# noise: [120, 1]
noise = torch.randn(num_train + num_val, 1) * 0.01

# labels: [120, 1]
labels = features @ true_weights + true_bias + noise


train_features = features[:num_train]
train_labels = labels[:num_train]

val_features = features[num_train:]
val_labels = labels[num_train:]


train_loader = DataLoader(
    dataset=TensorDataset(train_features, train_labels),
    batch_size=batch_size,
    shuffle=True,
)

print("Training:", train_features.shape, train_labels.shape)
print("Validation:", val_features.shape, val_labels.shape)

Training: torch.Size([20, 200]) torch.Size([20, 1])
Validation: torch.Size([100, 200]) torch.Size([100, 1])


In [ ]:
def squared_loss(predictions, labels):
    """미니배치의 평균 제곱손실을 계산한다."""
    return 0.5 * (predictions - labels).pow(2).mean()


def l2_penalty(weights):
    """가중치의 제곱된 L2 norm에 1/2를 곱한다."""
    return 0.5 * weights.pow(2).sum()

class LinearRegressionScratch:
    def __init__(self, num_inputs):
        
        # weight: [200, 1] + gradient
        self.weights = torch.randn(num_inputs, 1) * 0.01
        self.weights.requires_grad_()

        # 일반적으로 bias에는 weight decay를 적용하지 않는다.
        self.bias = torch.zeros(1, requires_grad=True)

    def __call__(self, features):
        # (batch_size, num_inputs) @ (num_inputs, 1)
        # -> (batch_size, 1)
        return features @ self.weights + self.bias

    def parameters(self):
        return [self.weights, self.bias]

In [ ]:
@torch.no_grad()
def evaluate(model, features, labels):
    predictions = model(features)
    return squared_loss(predictions, labels).item()


def train_scratch(lambd, num_epochs=100, learning_rate=0.01):
    model = LinearRegressionScratch(num_inputs)

    for epoch in range(num_epochs):
        for batch_features, batch_labels in train_loader:
            predictions = model(batch_features)

            # 데이터 손실과 L2 penalty를 합친 scalar loss다.
            loss = (
                squared_loss(predictions, batch_labels)
                + lambd * l2_penalty(model.weights)
            )

            # total loss가 weights와 bias에 대해 만드는 gradient를 계산한다.
            loss.backward()

            with torch.no_grad():
                for parameter in model.parameters():
                    parameter -= learning_rate * parameter.grad
                    parameter.grad.zero_()

    train_loss = evaluate(
        model,
        train_features,
        train_labels,
    )
    val_loss = evaluate(
        model,
        val_features,
        val_labels,
    )

    return model, train_loss, val_loss

In [ ]:
# lambda=0이므로 정규화를 사용하지 않는다.
model_without_decay, train_loss, val_loss =  train_scratch(lambd=0)

print("Without weight decay")
print("Training loss:", train_loss)
print("Validation loss:", val_loss)
print("Weight L2 norm:", model_without_decay.weights.norm().item())

Without weight decay
Training loss: 1.9000643072808465e-17
Validation loss: 0.019312694668769836
Weight L2 norm: 0.14704065024852753


In [ ]:
# lambda가 커질수록 weight를 0 방향으로 더 강하게 제한한다.
model_with_decay, train_loss, val_loss = train_scratch(lambd=3)

print("With weight decay")
print("Training loss:", train_loss)
print("Validation loss:", val_loss)
print("Weight L2 norm:", model_with_decay.weights.norm().item())

With weight decay
Training loss: 0.00035983993439003825
Validation loss: 0.009051363915205002
Weight L2 norm: 0.03057272732257843
